# Cambio Climático

> **Contexto de dominio:** [`docs/fuentes/cambio_climatico.md`](../../docs/fuentes/cambio_climatico.md)  
> **Bloque:** B | **Línea:** `cambio_climatico`  
> **Variable principal:** `temperatura` (°C)  
> **Modelos sugeridos:** SARIMA, SARIMAX, PROPHET, XGBOOST  
> Flujo: `Plan.md` sección 3 — ciclo estadístico completo.

**Antes de comenzar:** Leer `docs/fuentes/cambio_climatico.md` para entender:
- Variables ambientales clave y sus rangos físicos
- Normativa colombiana aplicable (umbrales normativos)
- Fuentes de datos oficiales y frecuencia de actualización
- Preguntas analíticas típicas de esta línea

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from estadistica_ambiental.io.loaders import load_csv
from estadistica_ambiental.io.validators import validate
from estadistica_ambiental.eda.variables import classify
from estadistica_ambiental.eda.quality import assess_quality
from estadistica_ambiental.eda.profiling import run_eda
from estadistica_ambiental.eda.viz import plot_series, plot_seasonal_means
from estadistica_ambiental.preprocessing.imputation import impute
from estadistica_ambiental.descriptive.univariate import summarize
from estadistica_ambiental.inference.stationarity import stationarity_report
from estadistica_ambiental.inference.trend import mann_kendall
from estadistica_ambiental.inference.intervals import exceedance_report
from estadistica_ambiental.features.climate import load_oni, enso_lagged
from estadistica_ambiental.config import ENSO_LAG_MESES
from estadistica_ambiental.predictive.registry import get_model, list_models
from estadistica_ambiental.evaluation.backtesting import walk_forward
from estadistica_ambiental.evaluation.comparison import rank_models
from estadistica_ambiental.config import DOCS_FUENTES

LINEA = "cambio_climatico"
VARIABLE = "temperatura"
UNIDAD = "°C"

print("Setup OK | Modelos disponibles:", list_models())

## 0b. Contexto de dominio
> Carga la ficha técnica de la línea `cambio_climatico` para tener presente la normativa, los indicadores y las preguntas analíticas durante el análisis.

In [ ]:
ficha = DOCS_FUENTES / "cambio_climatico.md"
if ficha.exists():
    print(ficha.read_text(encoding="utf-8")[:3000])  # primeras 3000 chars
else:
    print("Ficha no encontrada en", ficha)

## 1. Cargar datos
> Colocar el archivo en `data/raw/` y ajustar la ruta.  
> Ver sección **Datos y fuentes** de `docs/fuentes/cambio_climatico.md` para las fuentes oficiales.

In [ ]:
# df = load_csv("data/raw/cambio_climatico.csv", date_col="fecha")

# --- Datos sintéticos de ejemplo ---
np.random.seed(42)
n = 120
df = pd.DataFrame({
    "fecha":    pd.date_range("2015-01-01", periods=n, freq="ME"),
    "temperatura": np.random.gamma(3, 5, n) + np.linspace(0, 5, n),
})
print(f"Shape: {df.shape} | Rango: {df.fecha.min()} → {df.fecha.max()}")
df.head()

<!-- ENRICHMENT: cambio_climatico -->

## 1b. Contabilidad de Carbono — Inventario GEI por sector

Mientras el ONI captura el forzante climático externo, la **contabilidad de carbono** cuantifica
las emisiones de GEI internas del país. En Colombia el sector **AFOLU** representa ~60% de las
emisiones brutas — principalmente por deforestación en la Amazonia y los Andes.

| Registro | Gestiona | Referencia legal |
|---|---|---|
| **RENARE** | Reducciones de GEI (REDD+, MDL, NAMAs) | Ley 1753/2015, Res. 1447/2018 |
| **INGEI** | Inventario Nacional de GEI | IDEAM / Directrices IPCC |
| **SMByC** | Monitoreo de Bosques y Carbono | IDEAM — Satelital (Landsat/Sentinel) |

> **Meta NDC Colombia:** reducir 51% de emisiones netas para 2030 (tope: 169.440 kt CO2eq).

**Formula IPCC (Tier 1):**
```
Emisiones (tCO2e) = Datos_Actividad (AD) x Factor_Emision (EF) x PCG
Ej: ha_deforestadas x EF_biomasa_tC_ha x (44/12)  ->  tCO2e liberadas
```

In [ ]:
# Inventario GEI sintetico por sector (kt CO2e/ano)
# Estructura basada en INGEI Colombia 2015-2024
sectores = ['AFOLU', 'Energia', 'Transporte', 'Industria', 'Residuos']
n_anos = 10
anos = list(range(2015, 2015 + n_anos))

np.random.seed(99)
emisiones_kt = {
    'AFOLU':      [210 - i*2   + np.random.normal(0, 5)   for i in range(n_anos)],
    'Energia':    [65  + i*0.5 + np.random.normal(0, 2)   for i in range(n_anos)],
    'Transporte': [40  + i*0.8 + np.random.normal(0, 1.5) for i in range(n_anos)],
    'Industria':  [20  - i*0.1 + np.random.normal(0, 1)   for i in range(n_anos)],
    'Residuos':   [8   + i*0.2 + np.random.normal(0, 0.5) for i in range(n_anos)],
}
df_gei = pd.DataFrame({'ano': anos,
                        **{s: [round(v, 1) for v in vals]
                           for s, vals in emisiones_kt.items()}})
df_gei['total_kt'] = df_gei[sectores].sum(axis=1).round(1)

NDC_META_KT   = 169.4
NDC_BASE_2015 = float(df_gei.loc[df_gei['ano']==2015, 'total_kt'].values[0])

print(f'Linea base 2015: {NDC_BASE_2015:.1f} kt CO2e')
print(f'Meta NDC 2030:   {NDC_META_KT:.1f} kt CO2e (reduccion {(1-NDC_META_KT/NDC_BASE_2015)*100:.0f}%)')
df_gei

## 2. Validación y EDA
> `validate()` usa rangos físicos específicos para `cambio_climatico` desde `config.py`.

In [ ]:
val = validate(df, date_col="fecha", linea_tematica=LINEA)
print(val.summary())

In [ ]:
run_eda(df, output=f"data/output/reports/eda_cambio_climatico.html",
       title="EDA — Cambio Climático", date_col="fecha", use_ydata=False)
# Abrir el HTML en el navegador para el reporte completo

## 3. Visualización exploratoria

In [ ]:
plot_series(df, "fecha", "temperatura", title="Cambio Climático — temperatura (°C)")
plt.show()

In [ ]:
plot_seasonal_means(df, "fecha", "temperatura", period="month")
plt.show()

## 3c. Visualizacion NDC — Progreso y emisiones por sector

Los dos paneles responden las preguntas clave del ciclo MRV:
- **Izquierda:** evolucion de emisiones por sector — que sector lidera la reduccion?
- **Derecha:** a que distancia esta Colombia de la meta NDC del 51% para 2030?

> AFOLU (deforestacion) es la palanca principal: una reduccion del 30% en deforestacion
> equivale a ~60 kt CO2e menos — mas que todos los demas sectores combinados.

In [ ]:
colors_sect = ['#27ae60','#e74c3c','#3498db','#f39c12','#9b59b6']
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: emisiones stacked por sector
bottom = np.zeros(n_anos)
for sector, color in zip(sectores, colors_sect):
    ax1.bar(df_gei['ano'], df_gei[sector], bottom=bottom,
            label=sector, color=color, alpha=0.85, width=0.7)
    bottom += df_gei[sector].values
ax1.axhline(NDC_META_KT, color='black', ls='--', lw=2,
            label=f'Meta NDC 2030 ({NDC_META_KT} kt CO2e)')
ax1.set_title('Emisiones GEI por sector (kt CO2e/ano)', fontweight='bold')
ax1.set_ylabel('kt CO2e')
ax1.legend(fontsize=7, loc='upper right'); ax1.grid(axis='y', alpha=0.3)

# Panel B: progreso NDC (%)
pct_red = ((NDC_BASE_2015 - df_gei['total_kt']) / NDC_BASE_2015 * 100).clip(lower=0)
ax2.plot(df_gei['ano'], pct_red, marker='o', color='#27ae60', lw=2, label='Reduccion observada')
ax2.axhline(51, color='red', ls='--', lw=1.5, label='Meta NDC 51%')
ax2.fill_between(df_gei['ano'], pct_red, 51,
                 where=(pct_red < 51), alpha=0.15, color='red', label='Brecha pendiente')
ax2.set_title('Progreso NDC — % reduccion vs. linea base 2015', fontweight='bold')
ax2.set_ylabel('% reduccion'); ax2.set_ylim(0, 60)
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f'Reduccion {anos[-1]}: {pct_red.iloc[-1]:.1f}% | Brecha meta 51%: {51 - pct_red.iloc[-1]:.1f}pp')

## 3b. Covariable ENSO (ONI)
> Lag recomendado para `cambio_climatico` definido en `config.ENSO_LAG_MESES`.

In [ ]:
# --- Covariable ENSO (lag específico para Cambio Climático) ---
# Guard (M-20): avisar si LINEA no tiene lag específico — se aplicará el default.
if LINEA not in ENSO_LAG_MESES:
    _lag_default = ENSO_LAG_MESES["default"]
    print(f"[aviso] '{LINEA}' sin lag específico en ENSO_LAG_MESES; "
          f"se usará default ({_lag_default} meses).")
else:
    print(f"[ok] ENSO lag para '{LINEA}' = {ENSO_LAG_MESES[LINEA]} meses")

oni = load_oni()  # Descarga ONI desde NOAA
df = enso_lagged(df, oni, date_col="fecha", linea_tematica=LINEA)
print("Columnas ENSO agregadas:", [c for c in df.columns if "oni" in c or "enso" in c])

## 4. Estadística descriptiva

In [ ]:
summarize(df)

## 5. Inferencial
> ADR-004: Estacionariedad obligatoria pre-ARIMA (ADF + KPSS juntos).

In [ ]:
ts = df.set_index("fecha")["temperatura"].dropna()
stationarity_report(ts)

In [ ]:
mk = mann_kendall(ts)
print(f"Tendencia: {mk['trend']} | p={mk['pval']:.4f} | slope={mk['slope']:.6f}")

## 5b. Análisis de excedencias normativas
> Compara `temperatura` contra las normas colombianas relevantes.

In [ ]:
rep = exceedance_report(df["temperatura"], variable="temperatura")
if rep.empty:
    print("Sin normas colombianas registradas para 'temperatura'. "
          "Agregar umbral manual a la llamada exceedance_probability().")
else:
    display(rep)

## 6. Preprocesamiento

In [ ]:
df_clean = impute(df.copy(), cols=["temperatura"], method="linear")
print(f"Faltantes antes: {df["temperatura"].isna().sum()} | "
      f"después: {df_clean["temperatura"].isna().sum()}")

## 7. Modelos predictivos

In [ ]:
ts = df_clean.set_index("fecha")["temperatura"]

models = {
    "SARIMA":       get_model("sarima", order=(1,1,1), seasonal_order=(1,1,1,12)),
    "SARIMAX":      get_model("sarimax", order=(1,1,1), seasonal_order=(1,1,1,12)),
    "Prophet":      get_model("prophet"),
    "XGBoost":      get_model("xgboost", lags=[1,2,3,6,12]),
}

results = {}
for name, model in models.items():
    if name.startswith("#"):
        continue
    results[name] = walk_forward(model, ts, gap=12, horizon=6, n_splits=4)
    print(f"{name}: RMSE={results[name]['metrics']['rmse']:.3f}")

In [ ]:
rank_models(results)[["rmse","mae","r2","score","rank"]]

## 7b. Incertidumbre en factores de emision — Monte Carlo (Tier 1 vs. Tier 2)

Una de las principales criticas a los creditos REDD+ es el **over-crediting**: usar factores de
emision genericos (Tier 1, IPCC por defecto) en lugar de factores nacionales calibrados (Tier 2).
Monte Carlo cuantifica esa diferencia en MtCO2e con distribucion de probabilidad completa:

```
Emisiones = Area_deforestada x EF ~ Normal(mu, sigma)
n = 10.000 simulaciones
```

**Resultado clave:** Tier 2 reduce la incertidumbre porque usa datos del IFN
(Inventario Forestal Nacional) y parcelas del SMByC en lugar de promedios globales tropicales.
La diferencia de medias es el **over-crediting potencial** evitable con datos colombianos.

In [ ]:
np.random.seed(42)
N_SIM = 10_000

# Factor de emision AFOLU: biomasa aerea + subterranea (tCO2e/ha)
# Tier 1 (IPCC Tabla 4.7, bosque humedo tropical): mayor incertidumbre
# Tier 2 (SMByC/IFN Colombia): mas preciso, parcelas nacionales
EF_T1 = np.random.normal(120, 30, N_SIM)   # tCO2e/ha  +-25%
EF_T2 = np.random.normal(95,  12, N_SIM)   # tCO2e/ha  +-12%

AREA_HA = 50_000  # ha/ano de deforestacion
em_T1 = (AREA_HA * EF_T1 / 1e6).clip(min=0)  # MtCO2e
em_T2 = (AREA_HA * EF_T2 / 1e6).clip(min=0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(em_T1, bins=80, alpha=0.6, color='#e74c3c',
        label=f'Tier 1 (IPCC defecto) | media={em_T1.mean():.2f}, sigma={em_T1.std():.2f} MtCO2e')
ax.hist(em_T2, bins=80, alpha=0.6, color='#27ae60',
        label=f'Tier 2 (Colombia IFN) | media={em_T2.mean():.2f}, sigma={em_T2.std():.2f} MtCO2e')
ax.axvline(em_T1.mean(), color='#e74c3c', lw=2, ls='--')
ax.axvline(em_T2.mean(), color='#27ae60', lw=2, ls='--')
ax.set_title(f'Monte Carlo — Incertidumbre AFOLU (n={N_SIM:,} sim, area={AREA_HA:,} ha)',
             fontweight='bold')
ax.set_xlabel('MtCO2e/ano'); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

ci95_T1 = np.percentile(em_T1, [2.5, 97.5])
ci95_T2 = np.percentile(em_T2, [2.5, 97.5])
print(f'Tier 1 IC95%: [{ci95_T1[0]:.2f}, {ci95_T1[1]:.2f}] — rango={ci95_T1[1]-ci95_T1[0]:.2f} MtCO2e')
print(f'Tier 2 IC95%: [{ci95_T2[0]:.2f}, {ci95_T2[1]:.2f}] — rango={ci95_T2[1]-ci95_T2[0]:.2f} MtCO2e')
print(f'Reduccion de incertidumbre con Tier 2: {(1 - em_T2.std()/em_T1.std())*100:.0f}%')
print(f'Over-crediting potencial (Tier1-Tier2): {(em_T1.mean()-em_T2.mean()):.2f} MtCO2e')

## 7b. Guardrails y supuestos metodológicos
<!-- GUARDRAILS: cambio_climatico -->

> **Antes de publicar resultados**, verificar que se cumplen los supuestos clave del flujo. Esta sección lista los más comunes y los específicos de la línea.

### Supuestos comunes (todas las líneas)

- **Estacionariedad (ADR-004):** ADF + KPSS deben coincidir antes de aplicar ARIMA. Si discrepan, diferenciar conservadoramente o usar modelos no-ARIMA (Prophet, ML).
- **Outliers (ADR-002):** los picos ambientales son señal real (eventos, episodios). No aplicar clipping automático — sólo `preprocessing/outliers.py` opt-in y documentado.
- **Métrica primaria (ADR-003):** RMSLE NO en variables que pueden ser negativas o cero. Usar MAE + sMAPE (o NSE / KGE en hidrología) como default.
- **Tamaño muestral mínimo:** ARIMA requiere ≥ 36 observaciones; STL anual con datos diarios, ≥ 2 ciclos completos.
- **Residuos (post-fit):** verificar normalidad (Jarque-Bera) e independencia (Ljung-Box, lag = 12). Residuos correlacionados → modelo subespecificado.
- **Walk-forward con gap (BP-1):** series con ACF ≥ 0.7 inflan R². Usar `gap ≥ horizonte` en `walk_forward()`.
- **Normas oficiales:** usar `config.NORMA_*` y `config.*_THRESHOLDS` — nunca umbrales hardcodeados en el notebook (ADR-005).

### Supuestos específicos — Cambio climático / MRV

- **MRV (Medición, Reporte, Verificación):** trazabilidad obligatoria por compromisos NDC.
- **Agregados anuales:** los datos diarios introducen ruido — para tendencias decadales usar promedios anuales.
- **Lag ENSO = 0** (variable de contexto, sin lag específico).
- **Tendencias significativas:** requieren ≥ 30 años (criterio IPCC) — Mann-Kendall con muestras menores es ilustrativo, no concluyente.

### Antes de presentar a la autoridad ambiental

- Reportar intervalos de confianza, no sólo el punto estimado.
- Documentar el período de los datos, los gaps y el método de imputación usado.
- Registrar decisiones metodológicas no triviales en `docs/decisiones.md` (ADRs).


## 8. Conclusiones

- **Línea temática:** Cambio Climático (`cambio_climatico`)
- **Variable analizada:** `temperatura` (°C)
- **Modelos ejecutados:** SARIMA, SARIMAX, PROPHET, XGBOOST
- Completar con hallazgos reales al trabajar con datos de producción.

### Normativa y referencias
- Ver `docs/fuentes/cambio_climatico.md` para normativa colombiana, indicadores oficiales y fuentes de datos.
- Registrar decisiones metodológicas en `docs/decisiones.md`.